# Paper 3 — M1 Sim-to-Real Gate against DICHASUS cf02

Closes the M1 gate criterion that was provisional: the simulated delay-spread distribution is now
compared with the measured one from `dichasus_ref_stats.json` instead of a hand-set band.

No ray tracing: the M1 HDF5 already stores per-link RMS delay spread in `reports/sigma_tau`.

**The two environments are deliberately different** — DICHASUS cf02 is an indoor factory hall
(~14 x 14 m, arrays at the corners), the simulation is an outdoor urban block (~200 x 200 m).
A level shift is expected and is not a failure. The gate therefore tests *order of magnitude and
distribution shape*, with the criterion fixed below before the comparison is run.

In [ ]:
# %% [1] Setup
import os, json, glob
import numpy as np, h5py
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
BASE = '/content/drive/MyDrive/paper3'
M1_DIR = os.path.join(BASE, 'm1')
print([os.path.basename(p) for p in glob.glob(os.path.join(M1_DIR, '*.h5'))])

In [ ]:
# %% [2] Gate criterion — fixed BEFORE looking at the numbers
# (i)  the simulated median RMS delay spread is within a factor of 10 of the measured one
#      (indoor factory vs outdoor urban: a level shift is expected, an order-of-magnitude
#       disagreement would mean the scene or the material set is wrong);
# (ii) the distributions overlap: the sim P10-P90 range intersects the measured P10-P90 range;
# (iii) both distributions are right-skewed (median < mean), the signature of a multipath PDP.
FACTOR = 10.0

In [ ]:
# %% [3] Load simulated and measured delay spreads
h5s = sorted(glob.glob(os.path.join(M1_DIR, 'paper3_static_fr1_*.h5')))
sim_path = [p for p in h5s if 'A12' in p][0]        # the adopted geometry
with h5py.File(sim_path, 'r') as f:
    sigma = f['reports/sigma_tau'][:]                # [N, A] seconds
    members = f['cluster_gt/gamma'][:]               # serving-cluster mask
sim_ns = sigma[members]*1e9                          # serving-cluster links only
sim_ns = sim_ns[np.isfinite(sim_ns) & (sim_ns > 0)]

with open(os.path.join(BASE, 'dichasus_ref_stats.json')) as f:
    ref = json.load(f)
real_ns = np.array(ref['ds_ns'])
real_ns = real_ns[np.isfinite(real_ns) & (real_ns > 0)]
print(f'{os.path.basename(sim_path)}: {len(sim_ns)} simulated links')
print(f"DICHASUS {ref['dataset']}: {len(real_ns)} measured samples "
      f"(of {ref['n_samples']}, decimated by 10)")

In [ ]:
# %% [4] Comparison and gate
def describe(x, label):
    print(f'{label:10s} median {np.median(x):7.1f} ns | mean {x.mean():7.1f} | '
          f'P10 {np.percentile(x,10):7.1f} | P90 {np.percentile(x,90):7.1f}')
describe(sim_ns, 'simulated'); describe(real_ns, 'measured')

ratio = np.median(sim_ns)/np.median(real_ns)
lo_s, hi_s = np.percentile(sim_ns, [10, 90])
lo_r, hi_r = np.percentile(real_ns, [10, 90])
checks = {
    f'median within {FACTOR:.0f}x': 1/FACTOR <= ratio <= FACTOR,
    'P10-P90 ranges overlap': (lo_s <= hi_r) and (lo_r <= hi_s),
    'both right-skewed (median < mean)': (np.median(sim_ns) < sim_ns.mean())
                                         and (np.median(real_ns) < real_ns.mean()),
}
print(f'\nsim/real median ratio: {ratio:.2f}x')
for k, v in checks.items():
    print(f'[{"PASS" if v else "FAIL"}] {k}')
print('\nM1 SIM-TO-REAL GATE:', 'PASSED' if all(checks.values()) else 'NOT PASSED')

plt.figure(figsize=(6, 4))
for x, lab in ((sim_ns, 'simulation (Munich, A12, FR1)'), (real_ns, f"DICHASUS {ref['dataset']}")):
    xs = np.sort(x); plt.plot(xs, np.arange(1, len(xs)+1)/len(xs), label=lab)
plt.xscale('log'); plt.xlabel('RMS delay spread [ns]'); plt.ylabel('CDF')
plt.grid(alpha=0.3, which='both'); plt.legend(fontsize=8)
plt.title('Sim-to-real delay spread'); plt.tight_layout()
plt.savefig(os.path.join(M1_DIR, 'm1_gate_vs_dichasus.png'), dpi=150); plt.show()